# GAE — High-Dimensional Continuous Control Using Generalized Advantage Estimation (2015)

_Papers · Reinforcement Learning_

**An exponentially-weighted average of multi-step advantage estimates that slashes the variance of the policy gradient at the cost of a little bias.**

---

This notebook is a practice scaffold for the **GAE — High-Dimensional Continuous Control Using Generalized Advantage Estimation (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (Colab)

In [ ]:
# In Colab first run:  !pip install gymnasium
# (torch is preinstalled in Colab; gymnasium is not.)
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked 3-step rollout. ---
# gamma=0.9, lambda=0.8 (so gamma*lambda=0.72); episode ends at t=2, so V(s3)=0.
g, lam = 0.9, 0.8
r = [1.0, 0.0, 2.0]            # rewards
V = [1.0, 0.5, 1.0]           # critic values
Vn = [V[1], V[2], 0.0]        # next-state value (0 at terminal step)
done = [0.0, 0.0, 1.0]
delta = [r[t] + g * Vn[t] * (1 - done[t]) - V[t] for t in range(3)]   # Eq. 11
print("delta:", [round(d, 4) for d in delta])                         # [0.45, 0.4, 1.0]
gae, A = 0.0, [0.0, 0.0, 0.0]
for t in reversed(range(3)):                                          # Eq. 16 (recursive)
    gae = delta[t] + g * lam * (1 - done[t]) * gae
    A[t] = gae
print("A_GAE :", [round(a, 4) for a in A])                            # A[0] = 1.2564
G = 1.0 + 0.9 * 0.0 + 0.81 * 2.0                                      # discounted return at t=0
print("worked A_0^GAE =", round(A[0], 4),
      "| one-step (lam=0) =", round(delta[0], 4),
      "| Monte-Carlo (lam=1) = G - V =", round(G - V[0], 4))
# A_0^GAE = 1.2564  sits between  0.45 (lam=0)  and  1.62 (lam=1).


# --- 1. Policy + value networks (Track B: nn.Linear primitives). ---
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_act, hidden=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden), nn.Tanh(),
                                  nn.Linear(hidden, hidden), nn.Tanh())
        self.pi = nn.Linear(hidden, n_act)    # action logits -> policy
        self.v  = nn.Linear(hidden, 1)         # state value V(s) -> critic

    def forward(self, x):
        h = self.body(x)
        return Categorical(logits=self.pi(h)), self.v(h).squeeze(-1)


# --- 2. GAE advantage (Eq. 11 + Eq. 16) for a chosen lambda. ---
def compute_gae(rewards, values, dones, last_v, gamma=0.99, lam=0.95):
    n = len(rewards)
    adv = torch.zeros(n)
    values = values + [last_v]
    gae = 0.0
    for t in reversed(range(n)):
        mask  = 1.0 - dones[t]                                  # 0 if episode ended at t
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]   # Eq. 11
        gae   = delta + gamma * lam * mask * gae                # Eq. 16 (recursive form)
        adv[t] = gae
    return adv


# --- 3. Collect many short rollouts from a FIXED policy, then compare advantage variance. ---
def collect_first_step_advantages(n_rollouts=200, lambdas=(0.0, 0.95, 1.0),
                                  gamma=0.99, horizon=60):
    env = gym.make("CartPole-v1")
    net = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    net.eval()                                                  # policy + critic stay FIXED
    out = {lam: [] for lam in lambdas}
    for k in range(n_rollouts):
        obs, _ = env.reset(seed=1000 + k)
        R, Vs, D = [], [], []
        for _ in range(horizon):
            ot = torch.as_tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                dist, v = net(ot)
                a = dist.sample()
            obs, rew, term, trunc, _ = env.step(int(a))
            R.append(float(rew)); Vs.append(float(v)); D.append(1.0 if (term or trunc) else 0.0)
            if term or trunc:
                break
        with torch.no_grad():
            last_v = 0.0 if D[-1] == 1.0 else float(net(torch.as_tensor(obs, dtype=torch.float32))[1])
        for lam in lambdas:
            adv = compute_gae(R, Vs, D, last_v, gamma=gamma, lam=lam)
            out[lam].append(adv[0].item())                      # advantage of the FIRST action
    env.close()
    return out

torch.manual_seed(0)
adv_by_lambda = collect_first_step_advantages()
print("\nVariance of the first-step advantage A_0 across rollouts (fixed policy):")
for lam, vals in adv_by_lambda.items():
    t = torch.tensor(vals)
    label = {0.0: "one-step TD (lam=0)", 0.95: "GAE       (lam=0.95)", 1.0: "Monte-Carlo (lam=1)"}[lam]
    print(f"  {label}:  var = {t.var(unbiased=True).item():8.3f}")
# Variance RISES with lambda: the Monte-Carlo end (lam=1) is noisiest, the one-step end
# (lam=0) is smoothest, and GAE (lam=0.95) keeps most of the variance reduction.
# (Exact numbers vary by hardware/seed; our small run, not the paper's.)

## Visualize the data & results

_Does GAE give lower-variance advantage estimates than raw Monte-Carlo returns? We collect many short CartPole rollouts from a FIXED policy and critic, compute the first-step advantage A_0 for several lambda values, and plot how its variance grows as lambda goes from 0 (one-step TD) to 1 (Monte-Carlo)._

In [ ]:
# Sketch of how the curve above is produced (full loop in the CODE cell).
# Fix one randomly-initialized policy + critic (net.eval(), no training).
# For each of ~200 short CartPole rollouts, record rewards r_t and critic values V(s_t).
# For each lambda in a sweep, compute the first-step advantage with the GAE recursion:
#
#   delta[t] = r[t] + gamma*V[t+1]*mask - V[t]            # Eq. 11
#   gae      = delta[t] + gamma*lambda*mask*gae           # Eq. 16 (backward)
#   A_0      = gae at t=0
#
# Then take Var(A_0) across rollouts for each lambda and plot it.
#   lambda = 0    -> one-step TD  -> LOWEST variance (biased)
#   lambda = 1    -> Monte-Carlo  -> HIGHEST variance (unbiased)
#   lambda = 0.95 -> GAE          -> low variance, small bias  <-- the sweet spot
# (Numbers are from our own run; the paper reports MuJoCo locomotion results with
#  best gamma in [0.96,0.99], lambda in [0.92,0.98], not these CartPole values.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The worked rollout. With $\gamma = 0.9$, $\lambda = 0.8$, and the 3-step rollout
            $r = [1.0, 0.0, 2.0]$, $V = [1.0, 0.5, 1.0]$ (episode terminates at $t = 2$, so $V(s_3) = 0$),
            compute the three TD errors and then $\hat{A}_0^{GAE}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\delta_0 = 1.0 + 0.9\times 0.5 - 1.0 = 0.45$. — _TD error = reward + discounted next-value − current value (Eq. 11)._
- $\delta_1 = 0.0 + 0.9\times 1.0 - 0.5 = 0.40$. — _Same formula at $t=1$; $V(s_2) = 1.0$._
- $\delta_2 = 2.0 + 0.9\times 0 - 1.0 = 1.00$. — _Terminal step: $V(s_3) = 0$, no bootstrap._
- $\hat{A}_0 = 0.45 + 0.72\times 0.40 + 0.72^2\times 1.00 = 0.45 + 0.288 + 0.5184$. — _Eq. 16 with $\gamma\lambda = 0.9\times 0.8 = 0.72$; weight on $\delta_{0+l}$ is $(\gamma\lambda)^l$._

**Answer:** $\hat{A}_0^{GAE} = 1.2564$. The notebook recomputes $\delta = [0.45, 0.40, 1.00]$ and
                 $\hat{A}_0 = 1.2564$, which sits between the one-step value $\delta_0 = 0.45$ ($\lambda = 0$)
                 and the Monte-Carlo value $G_0 - V_0 = 2.62 - 1.0 = 1.62$ ($\lambda = 1$).

</details>

**Problem 2.** The variance ablation. You estimate $\hat{A}_0$ over many CartPole rollouts from a fixed
            policy, for $\lambda \in \{0, 0.95, 1\}$. Predict the ordering of the three variances, and
            explain what changing $\lambda$ does and why it matters for training.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $\hat{A}_0$ for each $\lambda$ on each rollout, keeping the policy and critic fixed so only $\lambda$ varies. — _An honest ablation changes exactly one thing — here $\lambda$ — so any difference in spread is attributable to it._
- Take the sample variance of $\hat{A}_0$ across rollouts for each $\lambda$. — _Variance is precisely the noisiness of the gradient signal that $\lambda$ is meant to control._
- Observe $\text{Var}(\lambda{=}0) \lt \text{Var}(\lambda{=}0.95) \lt \text{Var}(\lambda{=}1)$. — _Larger $\lambda$ keeps more far-future TD errors (weight $(\gamma\lambda)^l$ decays slower), and the far future is where the variance lives._

**Answer:** Variance increases with $\lambda$: the one-step estimate ($\lambda = 0$) is least noisy
                 but biased by the critic, the Monte-Carlo estimate ($\lambda = 1$) is unbiased but noisiest,
                 and GAE at $\lambda = 0.95$ sits in between &mdash; keeping most of the variance reduction while
                 paying only a little bias. That is why $\lambda \approx 0.95$ is the common default: a smoother
                 gradient learns faster and more stably than raw Monte-Carlo returns. The CODEVIZ panel shows
                 the variance climbing as $\lambda$ rises.

</details>

**Problem 3.** Suppose the critic $V$ is perfect (equals the true value function). What happens to the bias
            of GAE as you vary $\lambda$, and which $\lambda$ would you then prefer?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that bias in $\delta_t$ comes only from the critic's error. — _When $V = V^{\pi,\gamma}$, $\mathbb{E}[\delta_t] = A^{\pi,\gamma}$ exactly (the derivation's Step 1)._
- Note that if every $\delta_t$ is unbiased, every $(\gamma\lambda)$-weighted sum of them is too. — _A weighted sum of unbiased terms is unbiased regardless of the weights $\lambda$._
- Pick the $\lambda$ that minimizes variance, namely $\lambda = 0$. — _With no bias to trade away, you only want the lowest-variance estimate, which is the one-step TD ($\lambda = 0$)._

**Answer:** With a perfect critic, GAE is unbiased for every $\lambda$ &mdash; there is no
                 critic error to correct &mdash; so you would choose $\lambda = 0$ (the one-step TD advantage)
                 to get the lowest variance. The whole point of larger $\lambda$ is to lean less on an
                 imperfect critic; a perfect one removes that need. This is why GAE pairs $\lambda$ with
                 a value function that is itself being trained: while the critic is still wrong, a higher
                 $\lambda$ corrects its bias with real rewards.

</details>